PySpark and SEC EDGAR (Government Financial Data) %md


In [0]:
# Import PySpark SQL functions for data transformations (filtering, aggregating, etc.)
from pyspark.sql import functions as F
from pyspark.sql.types import *  # Import data types for defining schemas (StringType, IntegerType, etc.)
from pyspark.sql.window import Window  # Import for ranking and time-series operations

# Import Python libraries for API calls and parallel processing
import requests  # Make HTTP requests to SEC EDGAR API
from concurrent.futures import ThreadPoolExecutor  # Run multiple API calls in parallel
import time  # Add delays to respect SEC rate limits
from datetime import datetime  # Handle date/time operations

# ========================================
# WHY EXPLICIT CONFIGURATION MATTERS
# ========================================
# Production code principle: Always explicitly set critical configurations, even if they're defaults
# Reasons: (1) Makes code portable across different Spark versions and cluster setups
#          (2) Self-documents your intentions for anyone reading the code
#          (3) Protects against cluster-level overrides that might disable features
# ========================================

# ========================================
# ADAPTIVE QUERY EXECUTION (AQE)
# ========================================
# What it is: Feature that watches your query run and adjusts the execution plan in real-time
# How it works: After each stage completes, Spark collects actual statistics (row counts, data sizes)
#               and compares them to initial estimates. If there's a mismatch, it adjusts:
#               - Merges many small partitions into fewer larger ones (less overhead)
#               - Switches join strategies when one table is smaller than expected
#               - Splits oversized partitions to prevent one worker from being overwhelmed
# Why specify it: Even though it's default in Spark 3.2+, explicit setting ensures it works on
#                 older Spark versions (2.4, 3.0) and overrides any cluster-level disabling
# Why it matters here: SEC filing data is highly skewed (some companies file 100x more than others)
#                      AQE automatically handles this without manual partition tuning
# ========================================

# AQE-specific settings (lines below configure Adaptive Query Execution features)
spark.conf.set("spark.sql.adaptive.enabled", "true")  # Master switch: turns on AQE's real-time optimization engine
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")  # Partition merging: combines 100 tiny data chunks into 5 normal ones (reduces task overhead)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")  # Skew handling: splits Apple's 10K filings across multiple workers instead of overwhelming one (prevents bottlenecks)

# Why enable these specific AQE features?
# 1. Master switch (adaptive.enabled): Without this, the other two settings do nothing - it's the on/off button
# 2. Partition merging (coalescePartitions): SEC data varies wildly - one company might have 5 filings, another 500
#    Without merging, Spark creates too many tiny tasks for small companies, wasting time on coordination
# 3. Skew handling (skewJoin): When joining company info with filings, megacorps like Berkshire Hathaway have 1000x
#    more filings than a small startup. Without skew handling, one worker gets stuck processing Berkshire while
#    others sit idle. This feature splits the work evenly across all workers.

# General Spark settings (not AQE-specific)
spark.conf.set("spark.sql.shuffle.partitions", "200")  # Default partitions for distributed operations (groupBy, joins)

# Display settings for cleaner notebook output
spark.conf.set("spark.sql.repl.eagerEval.enabled", "true")  # Show DataFrames without calling .show()
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", "10")  # Limit automatic preview to 10 rows

print("✅ Setup complete! Spark configured with Adaptive Query Execution enabled.")
print(f"📊 Spark version: {spark.version}")
print(f"🔧 AQE Status: {spark.conf.get('spark.sql.adaptive.enabled')}")